# TF-IDF Toxicity Model Evaluation

This notebook loads the saved TF-IDF + logistic regression model, evaluates it on the validation split, and shows the key metrics and sample predictions.

In [1]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.metrics import classification_report, f1_score

from toxicity_detector.config.labels import LABELS_EN, TEXT_COL
from toxicity_detector.config.paths import UKR_PROCESSED

project_root = Path.cwd().resolve()
model_path = project_root / '..' / 'models' / 'tfidf_word_char_logreg_ovr.joblib'

if not model_path.exists():
    raise FileNotFoundError(f'Model not found at {model_path}')

print(f'Using model: {model_path}')

Using model: C:\Users\Amina\Documents\bachelor's\4-year\Thesis docs\Thesis project\PROJECT\notebooks\..\models\tfidf_word_char_logreg_ovr.joblib


In [2]:
test_df = pd.read_csv(UKR_PROCESSED['test'])

X_test = test_df[TEXT_COL].astype(str)
y_test = test_df[LABELS_EN].astype(int)

print(f'Validation rows: {len(test_df)}')
print(f'Label columns: {LABELS_EN}')

Validation rows: 2000
Label columns: ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']


In [3]:
model = joblib.load(model_path)

y_test_prob = model.predict_proba(X_test)
y_test_pred = (y_test_prob >= 0.5).astype(int)

macro_f1 = f1_score(y_test, y_test_pred, average='macro', zero_division=0)
micro_f1 = f1_score(y_test, y_test_pred, average='micro', zero_division=0)

print('Macro F1:', macro_f1)
print('Micro F1:', micro_f1)
print('\nPer-label report at threshold = 0.50:\n')
print(classification_report(y_test, y_test_pred, target_names=LABELS_EN, zero_division=0))

Macro F1: 0.6443628721384884
Micro F1: 0.6968325791855203

Per-label report at threshold = 0.50:

               precision    recall  f1-score   support

        toxic       0.76      0.79      0.78      1038
 severe_toxic       0.65      0.52      0.58       157
      obscene       0.67      0.70      0.69       595
       threat       0.55      0.69      0.61       188
       insult       0.74      0.69      0.71       597
identity_hate       0.57      0.44      0.50       309

    micro avg       0.70      0.69      0.70      2884
    macro avg       0.66      0.64      0.64      2884
 weighted avg       0.70      0.69      0.69      2884
  samples avg       0.39      0.36      0.36      2884



In [5]:
for i, label in enumerate(LABELS_EN):
    score = f1_score(y_test[label], y_test_pred[:, i], zero_division=0)
    print(f'{label}: {score:.4f}')

toxic: 0.7781
severe_toxic: 0.5775
obscene: 0.6863
threat: 0.6114
insult: 0.7129
identity_hate: 0.5000


## Interpretation

The model is evaluated using a 0.5 decision threshold for each toxicity label. The printed report gives the precision, recall, and F1-score for each label as well as the macro and micro averages.